## 1. Imports

In [1]:
import sys
sys.path.insert(0, "/Users/tiagoleite/Desktop/projeto_nlp")

In [2]:
import sys
import json
import os
import time
import pandas as pd

from pathlib                 import Path
from typing                  import Optional
from dotenv                  import load_dotenv
from google                  import genai
from google.genai            import types
from pydantic                import ValidationError
from sklearn.metrics         import (accuracy_score, classification_report, f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from src.schema              import EditalClassificado

## 2. Funções

In [3]:
# Lê um arquivo de prompt da pasta /prompts e retorna o texto

def carregarPrompt(nome_arquivo: str) -> str:
    caminho = Path("..") / "prompts" / nome_arquivo
    return caminho.read_text(encoding="utf-8")

In [4]:
# Substitui o marcador {texto} no template pelo texto do edital

def montarPrompt(prompt_template: str, texto: str) -> str:
    return prompt_template.replace("{texto}", texto)

In [5]:
# Chama a API do Gemini com o texto do edital e o prompt informado
# Valida a resposta com o schema Pydantic EditalClassificado
 
# foram criados os seguintes fluxos:
#    - Tentativa 1: chamada normal com o prompt original
#    - Tentativa 2 (retry): chamada com instrução mais explícita se a
#      validação falhar
#    - Se ambas falharem: registra o erro e retorna None

# parâmetros da função:
#        texto           — texto limpo do edital
#        prompt_template — conteúdo do arquivo de prompt (zero-shot ou few-shot)
#        cliente         — instância autenticada do cliente Gemini
#        modelo          — nome do modelo a ser usado
# 
# o que a funçao vai retornar:
#   - EditalClassificado validado ou None em caso de falha

def classificarEdital(texto: str,
                      prompt_template: str,
                      cliente: genai.Client,
                      modelo: str,) -> Optional[EditalClassificado]:
 
    prompt_retry = (
        "<instrucao_adicional>\n"
        "Sua resposta anterior não estava no formato correto.\n"
        "Responda SOMENTE com JSON puro, sem texto antes ou depois, "
        "sem blocos de código, sem markdown, sem explicações.\n"
        "O JSON deve conter exatamente estes campos:\n"
        "categoria_objeto (string), modalidade (pregão|concorrência|dispensa|inexigibilidade), "
        "valor_estimado (número ou null), requisitos_habilitacao (lista de strings), "
        "alerta_risco (baixo|médio|alto), oportunidade_recomendada (true|false).\n"
        "</instrucao_adicional>\n\n"
    )
 
    for tentativa in range(1, 3):
        try:
            if tentativa == 1:
                prompt_final = montarPrompt(prompt_template, texto)
            else:
                prompt_final = prompt_retry + montarPrompt(prompt_template, texto)
 
            resposta = cliente.models.generate_content(
                model=modelo,
                contents=prompt_final,
                config=types.GenerateContentConfig(
                    temperature=0,
                    response_mime_type="application/json",
                ),
            )
 
            texto_resposta = resposta.text.strip()
 
            # remove blocos de código se o modelo os incluir mesmo assim
            if texto_resposta.startswith("```"):
                linhas = texto_resposta.splitlines()
                linhas = [l for l in linhas if not l.startswith("```")]
                texto_resposta = "\n".join(linhas).strip()
 
            dados = json.loads(texto_resposta)
            resultado = EditalClassificado(**dados)
            return resultado
 
        except ValidationError as e:
            print(f"[tentativa {tentativa}] Erro de validação Pydantic: {e}")
            if tentativa == 2:
                print("[falha] Duas tentativas sem sucesso. Retornando None.")
                return None
 
        except json.JSONDecodeError as e:
            print(f"[tentativa {tentativa}] JSON inválido retornado pelo modelo: {e}")
            if tentativa == 2:
                print("[falha] Duas tentativas sem sucesso. Retornando None.")
                return None
 
        except Exception as e:
            print(f"[tentativa {tentativa}] Erro inesperado: {e}")
            if tentativa == 2:
                print("[falha] Duas tentativas sem sucesso. Retornando None.")
                return None
 
        time.sleep(6.5)
 
    return None

In [6]:
# a funçao roda classificarEdital em um lote de editais.
 
# os parâmetros:
#    df_amostra      — DataFrame com os editais a classificar
#    prompt_template — template do prompt (zero-shot ou few-shot)
#    cliente         — instância autenticada do cliente Gemini
#    modelo          — nome do modelo a ser usado
#    intervalo       — tempo de espera entre chamadas (em segundos)
 
# a funçao vai retornar:
#    Uma lista de dicionários com os resultados (ou None para falhas)
 

def classificarLote(df_amostra: pd.DataFrame,
                    prompt_template: str,
                    cliente: genai.Client,
                    modelo: str,
                    intervalo: float = 6.5) -> list:
    
    resultados = []
 
    for i, (idx, row) in enumerate(df_amostra.iterrows()):
        print(f"[{i+1}/{len(df_amostra)}] Classificando edital índice {idx}...")
 
        resultado = classificarEdital(
            texto=row["texto_limpo"],
            prompt_template=prompt_template,
            cliente=cliente,
            modelo=modelo,
        )
 
        if resultado:
            entrada = resultado.model_dump()
            entrada["idx_original"] = idx
            entrada["categoria_real"] = row["categoria"]
            resultados.append(entrada)
        else:
            resultados.append({
                "idx_original": idx,
                "categoria_real": row["categoria"],
                "modalidade": None,
                "erro": True,
            })
 
        time.sleep(intervalo)
 
    return resultados

In [7]:
# Converte o valor de modalidade do schema para o formato usado nas classes reais do dataset, para comparação com o baseline.
 
def mapearModalidade(modalidade: Optional[str]) -> Optional[str]:
    
    mapa = {
        "pregão": "Pregão - Eletrônico",
        "concorrência": "Concorrência - Eletrônica",
        "dispensa": "Dispensa de Licitação",
        "inexigibilidade": "Inexigibilidade",
    }
    if modalidade is None:
        return None
    return mapa.get(modalidade.lower(), None)

In [8]:
# Calcula F1 macro, precision, recall e accuracy para um conjunto de previsões.

def calcularMetricas(y_real: list, y_pred: list, nome_modelo: str) -> dict:

    return {
        "Modelo": nome_modelo,
        "F1 Macro": round(f1_score(y_real, y_pred, average="macro", zero_division=0), 4),
        "Precision": round(precision_score(y_real, y_pred, average="macro", zero_division=0), 4),
        "Recall": round(recall_score(y_real, y_pred, average="macro", zero_division=0), 4),
        "Accuracy": round(accuracy_score(y_real, y_pred), 4)}

In [9]:
#  Extrai os contadores de tokens da resposta da API do Gemini

def contarTokens(resposta) -> dict:
    uso = resposta.usage_metadata
    return {"tokens_entrada": uso.prompt_token_count,
            "tokens_saida": uso.candidates_token_count,
            "tokens_total": uso.total_token_count}

In [10]:
#Projeta o custo de processar n_editais com base na média de tokens coletada da API

def estimarCusto(media_entrada: float,
                 media_saida: float,
                 n_editais: int,
                 preco_entrada_por_milhao: float,
                 preco_saida_por_milhao: float,
                 nome_modelo: str) -> dict:
    
    total_entrada = media_entrada * n_editais
    total_saida = media_saida * n_editais
    custo_entrada = (total_entrada / 1_000_000) * preco_entrada_por_milhao
    custo_saida = (total_saida / 1_000_000) * preco_saida_por_milhao
    custo_total = custo_entrada + custo_saida
 
    return {
        "Modelo": nome_modelo,
        "Tokens entrada (média)": round(media_entrada, 1),
        "Tokens saída (média)": round(media_saida, 1),
        "Custo entrada USD": round(custo_entrada, 4),
        "Custo saída USD": round(custo_saida, 4),
        "Custo total USD": round(custo_total, 4),
        f"Custo total para {n_editais} editais (USD)": round(custo_total, 4),
    }

In [11]:
# Extrai listas de labels reais e previstos a partir dos resultados do lote e remove os registros com erro (modalidade None)

def extrairPrevisoes(resultados: list) -> tuple:
   
    y_real, y_pred = [], []

    for r in resultados:
        if r.get("modalidade") is None:
            continue
        pred_mapeada = mapearModalidade(r["modalidade"])
        if pred_mapeada is None:
            continue
        y_real.append(r["categoria_real"])
        y_pred.append(pred_mapeada)
        
    return y_real, y_pred

## 3. Cofiguração Gemini

In [21]:
load_dotenv()
 
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
 
if not GEMINI_API_KEY:
    raise ValueError("Chave GEMINI_API_KEY não encontrada no arquivo .env")
 
cliente = genai.Client(api_key=GEMINI_API_KEY)
 
MODELO_PRINCIPAL = "gemini-2.5-flash-lite"
 
print(f"Modelo principal: {MODELO_PRINCIPAL}")
print("Cliente Gemini configurado com sucesso.")

Modelo principal: gemini-2.5-flash-lite
Cliente Gemini configurado com sucesso.


## 4. Carga dos Dados

In [13]:
CAMINHO_PARQUET = Path("../dados/processed/editais_processado.parquet")
 
df = pd.read_parquet(CAMINHO_PARQUET)
 
print(f"Total de editais carregados: {len(df)}")
print(f"\nDistribuição por categoria:")
print(df["categoria"].value_counts())
print(f"\nColunas disponíveis: {df.columns.tolist()}")

Total de editais carregados: 3009

Distribuição por categoria:
categoria
Pregão - Eletrônico          1899
Concorrência - Eletrônica    1060
Concorrência - Presencial      50
Name: count, dtype: int64

Colunas disponíveis: ['modalidadeId', 'modoDisputaId', 'srp', 'dataAtualizacao', 'dataInclusao', 'dataPublicacaoPncp', 'orgaoEntidade', 'anoCompra', 'sequencialCompra', 'numeroCompra', 'unidadeOrgao', 'amparoLegal', 'dataAberturaProposta', 'dataEncerramentoProposta', 'informacaoComplementar', 'processo', 'objetoCompra', 'linkSistemaOrigem', 'justificativaPresencial', 'unidadeSubRogada', 'orgaoSubRogado', 'valorTotalHomologado', 'linkProcessoEletronico', 'emendaParlamentar', 'dataAtualizacaoGlobal', 'numeroControlePNCP', 'valorTotalEstimado', 'modalidadeNome', 'modoDisputaNome', 'tipoInstrumentoConvocatorioCodigo', 'tipoInstrumentoConvocatorioNome', 'fontesOrcamentarias', 'situacaoCompraId', 'situacaoCompraNome', 'usuarioNome', 'modalidade_nome', 'categoria', 'texto_limpo']


## 5. Schema Pydantic

In [14]:
# Exibe os campos do schema para conferência

for nome, campo in EditalClassificado.model_fields.items():
    print(f"  {nome}: {campo.annotation}")
    if campo.description:
        print(f"    → {campo.description[:80]}...")
    print()

  categoria_objeto: <class 'str'>
    → Categoria do objeto licitado, descrita em linguagem simples. Exemplos: 'serviços...

  modalidade: typing.Literal['pregão', 'concorrência', 'dispensa', 'inexigibilidade']
    → Modalidade do processo licitatório. Usar 'pregão' para Pregão Eletrônico ou Pres...

  valor_estimado: typing.Optional[float]
    → Valor estimado do contrato em reais (R$), como número decimal. Extrair do texto ...

  requisitos_habilitacao: typing.List[str]
    → Lista de exigências para o fornecedor participar da licitação. Exemplos: 'certid...

  alerta_risco: typing.Literal['baixo', 'médio', 'alto']
    → Nível de risco do edital para um fornecedor participar. 'baixo': objeto simples,...

  oportunidade_recomendada: <class 'bool'>
    → Indica se o edital é uma boa oportunidade para participar. True se o objeto é co...



## 6. Carga dos Prompts

In [15]:
prompt_zero_shot = carregarPrompt("prompt_zero_shot.txt")
prompt_few_shot = carregarPrompt("prompt_few_shot.txt")
 
print("Prompts carregados:")
print(f"  zero-shot: {len(prompt_zero_shot)} caracteres")
print(f"  few-shot:  {len(prompt_few_shot)} caracteres")


Prompts carregados:
  zero-shot: 1881 caracteres
  few-shot:  5037 caracteres


## 7. Testes

### 7.1 Selecionando Edital Manualmente

In [18]:
# Seleciona 10 editais variados, pelo menos um de cada categoria principal

CATEGORIAS_PRINCIPAIS = ["Pregão - Eletrônico",
                         "Concorrência - Eletrônica",
                         "Concorrência - Presencial"]

amostra_manual = (df[df["categoria"].isin(CATEGORIAS_PRINCIPAIS)].groupby("categoria", group_keys=False)
                                                             .sample(n=4, random_state=42)
                                                             .reset_index(drop=True))

print(f"Amostra Manual: {len(amostra_manual)} editais")
print(amostra_manual["categoria"].value_counts())

Amostra Manual: 12 editais
categoria
Concorrência - Eletrônica    4
Concorrência - Presencial    4
Pregão - Eletrônico          4
Name: count, dtype: int64


In [22]:
# roda o teste manual com o prompt zero-shot para saber se esta tudo funcionando
# O objetivo é 

resultados_manual = []
 
for i, row in amostra_manual.iterrows():
    print(f"[{i+1}/{len(amostra_manual)}] Categoria real: {row['categoria']}")
 
    resultado = classificarEdital(texto = row["texto_limpo"],
                                  prompt_template = prompt_zero_shot,
                                  cliente = cliente,
                                  modelo = MODELO_PRINCIPAL)
 
    if resultado:
        print(f"  modalidade classificada : {resultado.modalidade}")
        print(f"  categoria_objeto        : {resultado.categoria_objeto}")
        print(f"  valor_estimado          : {resultado.valor_estimado}")
        print(f"  alerta_risco            : {resultado.alerta_risco}")
        print(f"  oportunidade_recomendada: {resultado.oportunidade_recomendada}")
        print(f"  requisitos encontrados  : {len(resultado.requisitos_habilitacao)}")
    else:
        print("  FALHA: retornou None")
 
    resultados_manual.append({"categoria_real": row["categoria"],"resultado": resultado})
 
    print()

    time.sleep(6.5)

[1/12] Categoria real: Concorrência - Eletrônica
  modalidade classificada : concorrência
  categoria_objeto        : obra de reforma de unidade básica de saúde
  valor_estimado          : None
  alerta_risco            : médio
  oportunidade_recomendada: True
  requisitos encontrados  : 0

[2/12] Categoria real: Concorrência - Eletrônica
  modalidade classificada : concorrência
  categoria_objeto        : serviços de reforma de cemitério
  valor_estimado          : None
  alerta_risco            : médio
  oportunidade_recomendada: True
  requisitos encontrados  : 0

[3/12] Categoria real: Concorrência - Eletrônica
  modalidade classificada : concorrência
  categoria_objeto        : serviços especiais de engenharia e obras especiais
  valor_estimado          : None
  alerta_risco            : médio
  oportunidade_recomendada: False
  requisitos encontrados  : 0

[4/12] Categoria real: Concorrência - Eletrônica
  modalidade classificada : dispensa
  categoria_objeto        : serviços té

In [23]:
# resumo do teste manual

sucessos = sum(1 for r in resultados_manual if r["resultado"] is not None)
print(f"\nTeste manual concluído: {sucessos}/{len(resultados_manual)} com sucesso")


Teste manual concluído: 11/12 com sucesso


### 7.2 Zero-Shot

In [29]:

df_filtrado = df[df["texto_limpo"] != ""]

df_treino, df_teste = train_test_split(df_filtrado,
                                       test_size=0.30,
                                       random_state=42,
                                       stratify=df_filtrado["categoria"])

print(type(df_teste))
print(len(df_teste))

<class 'pandas.DataFrame'>
903


In [30]:
# filtra apenas as 3 classes principais
df_teste_principal = df_teste[df_teste["categoria"].isin(CATEGORIAS_PRINCIPAIS)].reset_index(drop=True)

print(f"Editais de teste nas 3 classes principais: {len(df_teste_principal)}")
print(df_teste_principal["categoria"].value_counts())


Editais de teste nas 3 classes principais: 903
categoria
Pregão - Eletrônico          570
Concorrência - Eletrônica    318
Concorrência - Presencial     15
Name: count, dtype: int64


In [32]:
# amostra estratificada de 150 editais
partes = []
for categoria, grupo in df_teste_principal.groupby("categoria"):
    n = min(int(150 * len(grupo) / len(df_teste_principal)), len(grupo))
    partes.append(grupo.sample(n, random_state=42))

df_amostra_teste = pd.concat(partes).head(150).reset_index(drop=True)

print(f"Amostra de teste: {len(df_amostra_teste)} editais")
print(df_amostra_teste["categoria"].value_counts())

Amostra de teste: 148 editais
categoria
Pregão - Eletrônico          94
Concorrência - Eletrônica    52
Concorrência - Presencial     2
Name: count, dtype: int64


In [33]:
# Classificaçao zero shot
 
resultados_zero_shot = classificarLote(df_amostra=df_amostra_teste,
                                       prompt_template=prompt_zero_shot,
                                       cliente=cliente,
                                       modelo=MODELO_PRINCIPAL)
 
print(f"\nZero-shot concluído.")
erros_zs = sum(1 for r in resultados_zero_shot if r.get("erro"))
print(f"Erros: {erros_zs}/{len(resultados_zero_shot)}")

[1/148] Classificando edital índice 0...
[2/148] Classificando edital índice 1...
[3/148] Classificando edital índice 2...
[4/148] Classificando edital índice 3...
[5/148] Classificando edital índice 4...
[6/148] Classificando edital índice 5...
[7/148] Classificando edital índice 6...
[8/148] Classificando edital índice 7...
[9/148] Classificando edital índice 8...
[tentativa 1] Erro inesperado: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 16.682113434s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Lear

### 7.3 Few-shot

In [ ]:
# usando os mesmos 150 editais

resultados_few_shot = classificarLote(df_amostra=df_amostra_teste,
                                      prompt_template=prompt_few_shot,
                                      cliente=cliente,
                                      modelo=MODELO_PRINCIPAL)
 
print(f"\nFew-shot concluído.")
erros_fs = sum(1 for r in resultados_few_shot if r.get("erro"))
print(f"Erros: {erros_fs}/{len(resultados_few_shot)}")

## 8. Métricas

In [ ]:
# previsões zero-shot
y_real_zs, y_pred_zs = extrairPrevisoes(resultados_zero_shot)
 
# previsões few-shot
y_real_fs, y_pred_fs = extrairPrevisoes(resultados_few_shot)

In [ ]:
# métricas dos dois prompts
metricas_zs = calcularMetricas(y_real_zs, y_pred_zs, "LLM zero-shot")
metricas_fs = calcularMetricas(y_real_fs, y_pred_fs, "LLM few-shot")
 
# métricas do baseline (valores registrados no notebook 1)
metricas_lr = {"Modelo": "Logistic Regression",
               "F1 Macro": 0.6471,
               "Precision": None,  # preencher com o valor do notebook 1 se disponível
               "Recall": None,
               "Accuracy": None}
 
metricas_svm = {"Modelo": "Linear SVM",
                "F1 Macro": 0.6020,
                "Precision": None,
                "Recall": None,
                "Accuracy": None}
 

In [ ]:
# tabela comparativa
tabela_comparativa = pd.DataFrame([ metricas_lr,
                                    metricas_svm,
                                    metricas_zs,
                                    metricas_fs])
 
print("=== TABELA COMPARATIVA DE MODELOS ===\n")
print(tabela_comparativa.to_string(index=False))
 
print("\n=== RELATÓRIO DETALHADO — ZERO-SHOT ===\n")
print(classification_report(y_real_zs, y_pred_zs, zero_division=0))
 
print("\n=== RELATÓRIO DETALHADO — FEW-SHOT ===\n")
print(classification_report(y_real_fs, y_pred_fs, zero_division=0))

## 9. Calculando o Custo

In [ ]:
# coleta tokens de 20 editais para calcular a média
AMOSTRA_CUSTO = 20
 
df_amostra_custo = df_amostra_teste.sample(AMOSTRA_CUSTO, random_state=42)
 
print(f"Coletando tokens de {AMOSTRA_CUSTO} editais para estimativa de custo...\n")
 
contagens = []
 
for i, (_, row) in enumerate(df_amostra_custo.iterrows()):
    try:
        prompt_final = montarPrompt(prompt_zero_shot, row["texto_limpo"])
 
        resposta = cliente.models.generate_content(model=MODELO_PRINCIPAL,
                                                   contents=prompt_final,
                                                   config=types.GenerateContentConfig(temperature=0,
                                                                                      response_mime_type="application/json"))
 
        tokens = contarTokens(resposta)
        contagens.append(tokens)
        print(f"[{i+1}/{AMOSTRA_CUSTO}] entrada: {tokens['tokens_entrada']} | saída: {tokens['tokens_saida']}")
 
    except Exception as e:
        print(f"[{i+1}/{AMOSTRA_CUSTO}] Erro ao contar tokens: {e}")
 
    time.sleep(1.5)
 
# médias
media_entrada = sum(c["tokens_entrada"] for c in contagens) / len(contagens)
media_saida = sum(c["tokens_saida"] for c in contagens) / len(contagens)
 
print(f"\nMédia de tokens de entrada: {media_entrada:.1f}")
print(f"Média de tokens de saída  : {media_saida:.1f}")
 

In [ ]:
# ATUALIZADO
# preços por 1M tokens (USD) — atualizado (Junho/2026)
# Linha principal atual (Geração 3 / 3.1) e Legados (2.5)

MODELOS_CUSTO = [
    # --- Geração Atual (Recomendada) ---
    {"nome": "gemini-3-flash", "preco_entrada": 0.50, "preco_saida": 3.00},
    {"nome": "gemini-3.1-flash-lite", "preco_entrada": 0.25, "preco_saida": 1.50},
    {"nome": "gemini-3.5-flash", "preco_entrada": 1.50, "preco_saida": 9.00}, # Lançado recentemente
    {"nome": "gemini-3.1-pro", "preco_entrada": 2.00, "preco_saida": 12.00}, # Contexto até 200k
    
    # --- Linha Legada (Legacy) ---
    {"nome": "gemini-2.5-flash", "preco_entrada": 0.30, "preco_saida": 2.50},
    {"nome": "gemini-2.5-flash-lite", "preco_entrada": 0.10, "preco_saida": 0.40}
]